In [3]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorboard
import wandb
import transformers
import lion_pytorch
import adam_mini
import copy
from tqdm.notebook import tqdm
import yaml
import timm 
import time
import torch.nn as nn
import torch.functional as F
from torchvision import transforms
from torch.utils.data import DataLoader, random_split
from torchvision.datasets import CIFAR10, CIFAR100
from torchvision import transforms
from torchvision.models import resnet18
from transformers.optimization import Adafactor
from lion_pytorch import Lion
from adam_mini import Adam_mini
from transformers import CLIPModel


In [6]:
def load_dataset(dataset_name="cifar10",batch_size= 16):
    #standard trasformation for this type of dataset
    mean = [0.4914, 0.4822, 0.4465]
    std = [0.2023, 0.1994, 0.2010]
    
    train_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomCrop(224, padding=4),
        transforms.ToTensor(),
        transforms.Normalize(mean, std)
    ])
    
    test_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean, std)
    ])

    if dataset_name.upper() == "CIFAR10":
        train = CIFAR10(
            root="./data",
            train=True,
            download=True,
            transform= train_transform
        )
        test = CIFAR10(
            root="./data",
            train=False,
            download=True,
            transform= test_transform
        )
    elif dataset_name.upper() == "CIFAR100":
        train = CIFAR100(
            root="./data",
            train=True,
            download=True,
            transform= train_transform
        )

        test = CIFAR100(
            root="./data",
            train=False,
            download=True,
            transform= test_transform
        )
    else:
        raise ValueError("Scegli tra 'CIFAR10' o 'CIFAR100'")
    
    train_size = int(0.8 * len(train))
    val_size = len(train) - train_size
    train_dataset, val_dataset = random_split(train, [train_size, val_size])
    train_dl= DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_dl= DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_dl= DataLoader(test, batch_size=batch_size, shuffle=False)


    return train_dl, val_dl, test_dl
        

In [7]:
def Model_definition(num_classes, model_name= "resnet" ):
    if model_name=="resnet":
        model = resnet18(
            weights=None,
            num_classes=10
        )
        in_features = model.fc.in_features
        model.fc = nn.Linear(in_features, num_classes)

    elif model_name=="":
        model = timm.create_model(
            "vit_tiny_patch16_224",
            pretrained=False,
            num_classes=num_classes
        )
    else:
        #da vedere
        model= None
    return model


In [ ]:
def format_elapsed_time(seconds):
    
    days = int(seconds // (24 * 3600))
    seconds %= (24 * 3600)
    hours = int(seconds // 3600)
    seconds %= 3600
    minutes = int(seconds // 60)
    secs = int(seconds % 60)
    parts = []
    if days > 0:
        parts.append(f"{days}d")
    if hours > 0 or days > 0:
        parts.append(f"{hours}h")
    if minutes > 0 or hours > 0 or days > 0:
        parts.append(f"{minutes}m")
    parts.append(f"{secs}s")
    
    time_str = " ".join(parts)
    
    return {
        "string": time_str,
        "days": days,
        "hours": hours,
        "minutes": minutes,
        "seconds": secs
    }


In [ ]:

class Earlystopping():
    def __init__(self, patience, min_delta = 0.005, ):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.best_model_wts = None


    def __call__(self, current_score, model):

        if self.best_score is None:
            self.best_score = current_score
            return True 
        improved = current_score > (self.best_score + self.min_delta)
        
        if improved:
            self.best_score = current_score
            self.counter = 0 
            return True
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
            return False
        


class Trainer(nn.Module):
    def __init__(self, config,wandb_run ):
        super(Trainer,self).__init__()
        self.name_model= config["name_model"]
        self.optimizer = config["optimizer"]
        self.batch_size = config["batch_size"]
        self.lr= config["lr"]
        self.weight_decay= config["weight_decay"]
        self.scheduler= config["scheduler"]
        self.epochs = config["epochs"]
        self.model= config["model"]
        self.patience= config["patience"]
        self.wandb= wandb_run
        self.criterion = nn.CrossEntropyLoss()
        self.device = "cuda:0" if torch.cuda.is_available() else "cpu"
        self.best_model= None
        self.val_accuracy = -np.inf 
        self.best_accuracy= 0   
        self.early_stopping = Earlystopping(self.patience)
    
    def train_one_epoch(self, dataloader):
        self.model.train()
        running_loss = 0
        total =0 
        correct = 0

        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats(self.device)
        
        start_time= time.time()
        for inputs, targets in dataloader:
            inputs, targets = inputs.to(self.device), targets.to(self.device)
            
            self.optimizer.zero_grad()
            outputs = self.model(inputs)
            loss = self.criterion(outputs, targets)
            loss.backward()
            self.optimizer.step()
            
            running_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
            
        end_time= time.time()-start_time
        epoch_loss = running_loss / total
        epoch_acc = 100 * correct / total
        peak_memory_mb = 0.0
        if torch.cuda.is_available():
            peak_memory_bytes = torch.cuda.max_memory_allocated(self.device)
            peak_memory_mb = peak_memory_bytes / (1024 ** 2)
        
        return epoch_loss, epoch_acc, peak_memory_mb, end_time
    
    
    @torch.no_grad()
    def evaluate(self,dataloader):
        self.model.eval()
        running_loss = 0.0
        correct = 0
        total = 0
        
        for inputs, targets in dataloader:
            inputs, targets = inputs.to(self.device), targets.to(self.device)
            outputs = self.model(inputs)
            loss = self.criterion(outputs, targets)
            
            running_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()

        peak_memory_mb = 0.0
        if torch.cuda.is_available():
            peak_memory_bytes = torch.cuda.max_memory_allocated(self.device)
            peak_memory_mb = peak_memory_bytes / (1024 ** 2)

        val_loss = running_loss / total
        val_acc = 100 * correct / total
        return val_loss, val_acc, peak_memory_mb
    


    def Training(self, dataloader_train, dataloader_test):
        self.best_accuracy=0
        start_time= time.time()
        for epoch in tqdm(self.epochs, desc= "Train Epochs"):

            train_loss, train_acc, train_mem, train_time= self.train_one_epoch(dataloader_train)
            val_loss, val_acc, val_mem = self.evaluate(dataloader = dataloader_test)

            time_data = format_elapsed_time(train_time)
            
            # 3. LOGGING SU WEIGHTS & BIASES (con metriche temporali complete)
            self.wandb.log({
                "epoch": epoch,
                "train/loss": train_loss,
                "train/accuracy": train_acc,
                "val/loss": val_loss,
                "val/accuracy": val_acc,
                "system/train_peak_gpu_memory_mb": train_mem,
                "system/val_peak_gpu_memory_mb": val_mem,
                "time_total/seconds": train_time,
                "time_total/formatted": time_data["string"],
                "time_total/days": time_data["days"],
                "time_total/hours": time_data["hours"],
                "time_total/minutes": time_data["minutes"],
                "time_total/seconds_remain": time_data["seconds"]
            })
            
            # Stampa di controllo
            print(f"Epoch {epoch:02d}/{self.epochs:02d} | "
                  f"Train Loss: {train_loss:.4f} - Acc: {train_acc:.2f}% | "
                  f"Val Loss: {val_loss:.4f} - Acc: {val_acc:.2f}% | "
                  f"Mem: {train_mem:.1f} MB | Time: {train_time:.2f}s")
            
            
            if  self.early_stopping:
                self.best_model = copy.deepcopy(self.model.state_dict())
                
            
            if self.scheduler is not None:
                self.scheduler.step()
                
        self.model.load_state_dict(self.best_model_wts)
        

        self.wandb.run.summary["best_val_accuracy"] = self.best_accuracy
        self.wandb.run.summary["total_training_time"] = format_elapsed_time(time.time() - start_time)["string"]
        
        
    
    def Test(self, dataloader):
        start_test= time.time()
        _,test_acc, test_mem = self.evaluate(dataloader = dataloader)
        end_test= time.time()- start_test
        time_data= format_elapsed_time(end_test)
        self.wandb.log({
            "test/accuracy_test": test_acc,
            "test/peak_gpu_memory_mb": test_mem,
            "test/time_total/formatted": time_data["string"],
            "test/time_total/days": time_data["days"],
            "test/time_total/hours": time_data["hours"],
            "test/time_total/minutes": time_data["minutes"],
            "test/time_total/seconds_remain": time_data["seconds"]
        })

